> **Código académico, sin garantía.** Material de los cursos de Modelado y Diseño de Líneas de Transmisión (Universidad Nacional de Colombia). Se entrega tal cual, sin garantía de ninguna clase; **no debe usarse para decisiones de diseño, operación o seguridad de instalaciones reales** sin verificación independiente. Ver [`DISCLAIMER.md`](../../DISCLAIMER.md). Licencia por definir (intención: código abierto).
----
> *Código realizado con apoyo de Claude y revisado por Ernesto Pérez para apoyo académico.*


# Taller: gradiente de campo eléctrico y efecto corona

Este notebook es el apoyo para realizar las preguntas del taller `Taller_corona.docx`; Con esto se busca que puedan a partir del análisis de sensibilidad en diferentes variables entender el comportamiento del efecto corona y poder tomar el criterio para decidir el número de haz de conductores analizando las variables de campo eléctrico por diferentes medios.


Cada pregunta trae una celda con un ejemplo de **dos puntos** y usted agrega los valores que necesite para sustentar su respuesta.

## Cómo se usa

1. En las ección 1 ajuste a los parámetros de línea de su ejemplo, y la sección 2 no la cambie y serán los resultados base. Dan la línea base y sus resultados, pero no necesariamente es la solución.
2. En cada pregunta, lea qué parámetro conviene variar, agregue más valores a la lista de la celda de ejemplo y ejecútela de nuevo.
3. Responda en el Word con sus palabras. Pegue la tabla o la figura que sustenta cada respuesta: guarde la figura con `fig.savefig("p1.png")` y la tabla con `tabla.to_csv("p1.csv")`.

## Qué se puede variar y cómo

Todos los parámetros están en el objeto `CASE` de la sección 1. 
Para ver el efecto de un cambio en alguna variable sin modificar la línea base se usan tres funciones:

| Función | Qué hace | Ejemplo |
|---|---|---|
| `analyze(CASE, ...)` | resultados de la línea con uno o más parámetros cambiados | `analyze(CASE, spacing=0.30)` |
| `sweep(CASE, nombre, valores)` | una fila de resultados por cada valor de la lista | `sweep(CASE, "n_sub", [2, 4])` |
| `plot_sweep(tabla, columnas)` | gráfica de una o más columnas contra el valor variado | `plot_sweep(tabla, ["E_max b (simulación) [kV/cm]"])` |

Si la variación cambia varios parámetros a la vez, `sweep` acepta una función `make` que recibe el valor y devuelve los parámetros que cambian:

```python
sweep(CASE, "altura de los guardas [m]", [30.0, 40.0], make=lambda h: {"guards": tuple((gx, h) for gx, _ in CASE.guards)})
```

| Parámetro | Significado | Unidad |
|---|---|---|
| `v_ll_kv` | tensión de línea, valor eficaz | kV |
| `phases` | posición `(x, y)` del centro del haz de las fases a, b, c | m |
| `n_sub`, `spacing`, `r_sub` | subconductores por fase, separación entre contiguos y radio de uno | -, m, m |
| `guards`, `r_guard`, `guard_mode` | posición y radio de los cables de guarda; `"grounded"`, `"insulated"` o `"none"` | m, m |
| `altitude_m`, `temp_c`, `m_surface` | altitud, temperatura y estado de la superficie del cable (Peek) | m, °C, - |

Un cambio de un parámetro con nombre equivocado da un error, así que no hay riesgo de variar algo por accidente.

**Cómo leer las columnas de las tablas.** El nombre de cada columna dice cómo se calculó: `(simulación)` es la simulación de cargas, `(Kuffel)` es la fórmula de Kuffel / CIGRE para el gradiente del haz y `(Peek)` es la fórmula del gradiente crítico. `E_max` sale de la simulación o de la fórmula de Kuffel, según la columna; `E_c` siempre sale de la fórmula de Peek.

## Dos tipos de cálculo: simulación de cargas y fórmulas

Los resultados de este taller salen de dos tipos de cálculo. Cada tabla y cada figura dice cuál se usó: `(simulación)` o `simulación de cargas` para el primero; `(Kuffel)`, `(Peek)` o `fórmula` para el segundo.

| | Simulación de cargas | Fórmulas |
|---|---|---|
| Qué es | Método numérico. Cada conductor se reemplaza por varias **cargas lineales ficticias** ubicadas dentro de él. Se calcula su valor para que la superficie del conductor quede a su tensión (y los cables de guarda a tierra, a 0 V). Con esas cargas se obtiene el campo en cualquier punto. | Ecuaciones cerradas de la literatura, que se evalúan directamente. |
| Qué entrega aquí | El gradiente máximo `E_max` en la superficie de cada subconductor, el campo al nivel del suelo y el campo y las equipotenciales de las figuras y animaciones. | El gradiente crítico `E_c` (Peek), un valor aproximado de `E_max` (Kuffel / CIGRE, a partir de la carga de la fase) y las ecuaciones del libro de CIGRE. |
| Exactitud | Usa la geometría exacta: todas las fases, los guardas y la tierra. | Hacen simplificaciones, por ejemplo ignoran el campo de las otras fases. |

Las cargas ficticias (`N_CHARGES` por conductor) son una herramienta del método y no son cargas reales de la línea. La carga real de una fase, por unidad de longitud, es la suma de las cargas ficticias de todos sus subconductores.

In [ ]:
import pathlib
import sys

# Se busca la raíz del repositorio (la carpeta que contiene src/) para que el notebook corra desde cualquier lugar.
root = pathlib.Path.cwd().resolve()
while not (root / "src" / "electric_field").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root / "src" / "electric_field"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import importlib
import corona
import charge_simulation
import line_case

# Recarga los módulos para usar siempre la versión actual del código, aunque el kernel ya los hubiera importado.
for _module in (corona, charge_simulation, line_case):
    importlib.reload(_module)

# Carga las librerías de la carpeta src/electric_field desarrolladas para este proyecto
from charge_simulation import plot_layout
from line_case import LineCase, analyze, build, ground_profile, plot_sweep, sweep

plt.rcParams.update({"figure.dpi": 110, "font.size": 11})

## 1. La línea base

Modifique la línea base al tipo de línea y configuración de conductores que usted tiene. 
En este caso se usó una línea trifásica de 500 kV, con un haz de 3 subconductores por fase y dos cables de guarda conectados a tierra, a 1500 m de altitud. 

In [ ]:
CASE = LineCase(
    v_ll_kv=115.0,                                   # line-to-line rms voltage [kV]
    phases=((-7.0, 15.0), (0.0, 15.0), (7.0, 15.0)),   # (x, y) of the bundle centre of phases a, b, c [m]
    n_sub=1,                                         # sub-conductors per phase
    spacing=0.45,                                    # distance between adjacent sub-conductors [m]
    r_sub=0.0141,                                    # radius of ONE sub-conductor [m]
    guards=((-4.0, 22.0), (4.0, 22.0)),              # shield wires (x, y) [m]; () for none
    r_guard=0.0055,                                  # shield wire radius [m]
    guard_mode="grounded",                           # "grounded" (0 V), "insulated" or "none"
    altitude_m=1500.0,                               # [m]
    temp_c=20.0,                                     # [degC]
    m_surface=0.8,                                   # Peek surface factor (0.8 dry stranded cable)
)

## 2. Diagrama y resultados de la línea base

El diagrama muestra la sección transversal (los conductores se dibujan más gruesos para que se vean) y, a la derecha, el haz de la fase b a escala. La tabla da el gradiente máximo sobre la superficie de cada fase (valor pico, kV/cm), calculado con la simulación de cargas y, para comparar, con la fórmula de Kuffel; el gradiente crítico `E_c` de la fórmula de Peek y el margen `E_max / E_c`, que debe ser menor que 0,95. Al final, el campo eléctrico más alto a 1 m del suelo (valor eficaz, kV/m), calculado con la simulación de cargas, y su posición.

In [ ]:
# Esta sección dibuja la configuración de la línea y calcula los parámetros de corona iniciales
# ****: Verifique que los parámetros de CASE sean correctos antes de continuar.
#
#  -no modifique nada debajo de esta línea-
wires, _ = build(CASE)
plot_layout(wires, guard_groups=(3, 4), zoom_group=1)
pd.Series(analyze(CASE), name="línea base").to_frame().round(3)

## 3. Preguntas

Cada celda de ejemplo usa dos puntos. **Agregue los valores que necesite** a las listas marcadas con `<-` y ejecute de nuevo. Las respuestas van en el Word.

### Pregunta 1: número de conductores

Parámetro que se varía: `n_sub`. El ejemplo compara 1 y 2 subconductores por fase; agregue otros valores.

In [ ]:
values = [1,2]   #   <- agregue más valores de n_sub separados por comas para hacer un barrido más amplio
# -- No cambiar nada debajo de esta línea --
table1 = sweep(CASE, "n_sub", values)
fig = plot_sweep(table1, ["E_max b (simulación) [kV/cm]"], title="Gradiente de la fase b contra subconductores por fase")
table1[["E_max b (simulación) [kV/cm]", "E_c (Peek) [kV/cm]", "E_max / E_c, peor fase (simulación)", "cumple < 0,95 (simulación)"]].round(2)

### Pregunta 2: fase con el mayor gradiente

La tabla de la sección 2 ya da el gradiente de cada fase de la línea base. Para comprobar su explicación, compare con una variación de la geometría: el ejemplo la compara con otra geometría (fases a 25 m de altura y 11 m de distancia). Cambie las posiciones de `phases` por otras.

In [ ]:
base = analyze(CASE)
variation = analyze(CASE, phases=((-11.0, 25.0), (0.0, 25.0), (11.0, 25.0)))      # <- cambie las posiciones (x, y)

#  -no modifique nada debajo de esta línea-
rows = ["E_max a (simulación) [kV/cm]", "E_max b (simulación) [kV/cm]", "E_max c (simulación) [kV/cm]", "peor fase (simulación)"]
table2 = pd.DataFrame({"base": base, "variación": variation}).loc[rows]
table2

### Pregunta 3: distancia entre conductores

Hay dos distancias que se pueden variar: la separación entre los subconductores de un haz (`spacing`) y la distancia entre fases (`phases`). Haga las dos. Cada ejemplo usa dos valores.

Si su línea tiene un solo conductor por fase (`n_sub=1`), la separación entre subconductores (`spacing`) no tiene efecto: para esa variación use `n_sub` de 2 o más.

Hay además una tercera variación: al aumentar el número de subconductores se reduce el radio de cada uno, porque la corriente por subconductor es menor. En el ejemplo se mantiene constante la sección total por fase (`n_sub · r_sub²`); puede cambiar esa regla.

In [ ]:
values = [0.45, 0.5]  # <- separación entre subconductores [m]; agregue más separadas por comas para hacer un barrido más amplio
# -no modifique nada debajo de esta línea-
table3a = sweep(CASE, "spacing", values)
fig = plot_sweep(table3a, ["E_max a (simulación) [kV/cm]", "E_max b (simulación) [kV/cm]"], title="Gradiente contra separación entre subconductores")
table3a[["E_max a (simulación) [kV/cm]", "E_max b (simulación) [kV/cm]", "E_max / E_c, peor fase (simulación)"]].round(2)

In [ ]:
distances = [9.0,9.1]     # <- distancia entre fases [m]; agregue más
#  -no modifique nada debajo de esta línea-
# se mantienen las alturas de las fases de la línea base; solo cambia la distancia entre ellas
heights = [p[1] for p in CASE.phases]
table3b = sweep(CASE, "distancia entre fases [m]", distances,
                make=lambda d: {"phases": ((-d, heights[0]), (0.0, heights[1]), (d, heights[2]))})
fig = plot_sweep(table3b, ["E_max a (simulación) [kV/cm]", "E_max b (simulación) [kV/cm]"], title="Gradiente contra distancia entre fases")
table3b[["E_max a (simulación) [kV/cm]", "E_max b (simulación) [kV/cm]", "E_max / E_c, peor fase (simulación)"]].round(2)

In [ ]:
# Variación 3: al aumentar el número de subconductores se reduce el radio de cada uno. Aquí se mantiene
# constante la sección total por fase (n_sub * r_sub^2 igual a la de la línea base).
values = [1, 2]        # <- número de subconductores; agregue más
#  -no modifique nada debajo de esta línea-
radius_for = lambda n: CASE.r_sub * (CASE.n_sub / n) ** 0.5
table3c = sweep(CASE, "número de subconductores", values, make=lambda n: {"n_sub": n, "r_sub": radius_for(n)})
table3c["radio del subconductor [cm]"] = [100 * radius_for(n) for n in values]
fig = plot_sweep(table3c, ["E_max b (simulación) [kV/cm]"], title="Gradiente con radio decreciente al aumentar los subconductores")
table3c[["radio del subconductor [cm]", "E_max b (simulación) [kV/cm]", "E_c (Peek) [kV/cm]",
         "E_max / E_c, peor fase (simulación)", "cumple < 0,95 (simulación)"]].round(2)

### Pregunta 4: campo eléctrico en el suelo

`ground_profile` da el campo vertical a 1 m de altura (valor eficaz, kV/m) a lo ancho de la línea. El ejemplo dibuja la línea base y la compara con una distancia entre fases de 10 m. El campo se calcula con la simulación de cargas. Las líneas verticales punteadas son las fases de la línea base.

In [ ]:
x, e = ground_profile(CASE)
x2, e2 = ground_profile(CASE, phases=((-10.0, 15.0), (0.0, 15.0), (10.0, 15.0)))    # <- cambie la variación

#  -no modifique nada debajo de esta línea-
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(x, e, lw=2, label="base")
ax.plot(x2, e2, lw=2, label="variación")
for px, _ in CASE.phases:
    ax.axvline(px, color="0.6", ls=":")
ax.set_xlabel("x [m]")
ax.set_ylabel("|E_y| rms a 1 m [kV/m] (simulación de cargas)")
ax.legend()
ax.grid(alpha=0.3)
print(f"Base: máximo {e.max():.2f} kV/m rms en x = ±{abs(x[e.argmax()]):.1f} m")
print(f"Variación: máximo {e2.max():.2f} kV/m rms en x = ±{abs(x2[e2.argmax()]):.1f} m")

### Pregunta 5: cables de guarda

Se compara la línea con guardas a tierra contra la misma línea sin guardas (`guard_mode="none"`). El ejemplo varía la altura de los guardas con dos valores; la distancia horizontal de los guardas al centro también se puede variar cambiando la `x` de cada guarda en `guards`. El efecto es pequeño, así que mire el cambio en porcentaje.

In [ ]:
without = analyze(CASE, guard_mode="none")["E_max b (simulación) [kV/cm]"]
guard_heights = [30.0, 40.0]                             # <- altura de los guardas [m]; agregue más
# se mantiene la posición horizontal (x) de los guardas de la línea base; solo cambia su altura
table5 = sweep(CASE, "altura de los guardas [m]", guard_heights,
               make=lambda h: {"guards": tuple((gx, h) for gx, _ in CASE.guards)})
table5["cambio respecto de la línea sin guardas [%]"] = 100 * (table5["E_max b (simulación) [kV/cm]"] / without - 1)
print(f"Sin cables de guarda: E_max b (simulación) = {without:.3f} kV/cm")
table5[["E_max b (simulación) [kV/cm]", "cambio respecto de la línea sin guardas [%]"]].round(3)

### Pregunta 6: simulación de cargas contra fórmulas

La pregunta pide decidir si las ecuaciones aproximadas bastan para escoger el número de subconductores y la posición de los cables. Cada fila de `analyze` trae el gradiente de la fase calculado de dos formas: con la simulación de cargas y con la fórmula de Kuffel, que no simula nada. Las columnas `cumple` dicen si cada método pasa el criterio de 0,95. Compare las dos decisiones y la diferencia en porcentaje en las variaciones que hizo en las preguntas 1, 3 y 5, y pruebe una geometría que ponga las fases más cerca entre sí.

In [ ]:
values = [1, 2]        # <- número de subconductores; puede cambiar a otro parámetro con sweep(CASE, "nombre", valores)
#  -no modifique nada debajo de esta línea-
table6 = sweep(CASE, "n_sub", values)
fig = plot_sweep(table6, ["E_max b (simulación) [kV/cm]", "E_max b (Kuffel) [kV/cm]"],
                 title="Gradiente de la fase b: simulación de cargas y fórmula de Kuffel")
table6[["E_max b (simulación) [kV/cm]", "E_max b (Kuffel) [kV/cm]", "Kuffel vs simulación, peor fase [%]",
        "cumple < 0,95 (simulación)", "cumple < 0,95 (Kuffel)"]].round(2)

## 4. Guardar los resultados

Para pegar en el Word: guarde la figura de la celda que acaba de ejecutar con `fig.savefig("p1.png", dpi=150)` y la tabla con `table1.to_csv("p1.csv")`. Los archivos quedan en la misma carpeta que este notebook.